## Validation Script for Porting the Nugraph DA code 

#### Set autoreloading
This extension will automatically update with any changes to packages in real time

In [ ]:
%load_ext autoreload
%autoreload 2

#### Append the path of the Nugraph Base conda libraries

In [ ]:
import os, sys
sys.path.append('/home/twalton/.conda/envs/NugraphBase/lib/python3.10/site-packages')

#### Import packages for training

In [ ]:
from pathlib import Path
import nugraph as ng
import pytorch_lightning as pl
print(ng.__file__)

#### Determine the run configuration 

In [ ]:
input_files = 1 #1 or 2

model_in_features = 5 if input_files == 1 else 4
model_da_loss_fnc_name = None if input_files == 1 else "dann"
model_warmup_epochs = 0 if input_files == 1 else 10

#### Set the data and module to use

In [ ]:
Data  = ng.data.NuGraphDataModule
Model = ng.models.NuGraph3

#### Declare and configure the data module

In [ ]:
source_filename="/scratch/7DayLifetime/cerati/concat-final-makeup.tiny.gnn.h5"
target_filename="/scratch/7DayLifetime/cerati/icarus_numi_2d.gnn.h5"

if input_files == 1:
   nudata = Data(model=Model, data_source_path=source_filename)
elif input_files == 2:
   nudata = Data(model=Model, data_source_path=source_filename, data_target_path=target_filename)

nudata.event_classes = ['cc_nue', 'cc_numu', 'cc_nutau', 'nc']
print(nudata)
print(nudata.semantic_classes)
print(nudata.event_classes)

#### Configure network

In [ ]:
nugraph = Model(
    in_features=model_in_features,  
    hit_features=128,
    nexus_features=32,
    instance_features=32,
    interaction_features=32,
    semantic_classes=nudata.semantic_classes, 
    event_classes=nudata.event_classes,
    num_iters=5,
    event_head=True,
    semantic_head=False,
    filter_head=False,
    vertex_head=False,
    instance_head=False,
    use_checkpointing=True,
    lr=0.001,
    da_loss_fnc_name=model_da_loss_fnc_name,
    warmup_epochs=model_warmup_epochs)

#### Configure logger and callbacks
Declare a TensorBoard logger and define the output directory, so we can monitor network training. Also, define a callback so we can monitor learning rate evolution.

In [ ]:
from pytorch_lightning.loggers import TensorBoardLogger
from datetime import datetime
now = datetime.now()
folder_name = "uBooNE-Tiny-Data-%s" % now.strftime("%Y-%m-%d-%H")

In [ ]:
os.environ["NUGRAPH_LOG"]='/home/twalton/NuGraphLogs/NuGraphMain/' 
logdir = Path(os.environ["NUGRAPH_LOG"])
logdir.mkdir(parents=True, exist_ok=True)
logger = TensorBoardLogger(save_dir=logdir,name=folder_name) 
callbacks = [
    pl.callbacks.LearningRateMonitor(logging_interval="step"),
    pl.callbacks.ModelCheckpoint(monitor="loss/val", mode="min"),
]

#### Declare trainer and run training
First, we set the training device. To train with a GPU, pass an integer; otherwise, it defaults to CPU training. We then instantiate a PyTorch Lightning trainer and run the training stage, iterating over all batches in the training and validation datasets to optimize model parameters, logging metrics to TensorBoard.

In [ ]:
device_number = 0
accelerator, devices = ng.util.configure_device(device_number)
print(accelerator)
print(devices)

trainer = pl.Trainer(accelerator=accelerator,
                     devices=devices,
                     max_epochs=1,
                     logger=logger,
                     callbacks=callbacks)
trainer.fit(nugraph, datamodule=nudata)
trainer.test(datamodule=nudata)